# RDFNet Baseline Evaluation

Evaluates the RDFNet baseline checkpoint (mAP) on the **FDD**, **VOC-FOG**, and **RTTS** test sets, using the code from `github.com/habibour/adverse_weather_ibject_detection`.

### Before running
1. In the right sidebar, click **+ Add Data** and attach these 4 datasets:
   - `mdhabibourrahman/fdd-dataset`
   - `mdhabibourrahman/voc-fog`
   - `tuncnguyn/rtts-dataset`
   - `mdhabibourrahman/rdfnet-baseline`
2. **Settings > Accelerator > GPU T4 x2** (or any available GPU).
3. Run all cells top to bottom. The first few cells only clone code and print folder trees — nothing expensive happens until the `evaluate(...)` calls near the bottom.

In [ ]:
%cd /kaggle/working
!rm -rf adverse_weather_ibject_detection
!git clone --depth 1 https://github.com/habibour/adverse_weather_ibject_detection.git
%cd /kaggle/working/adverse_weather_ibject_detection/RDFNet

In [ ]:
!pip install -q opencv-python-headless tqdm thop
import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## Inspect attached datasets

The evaluation code (`get_map.py`) expects each dataset laid out as:
```
<root>/VOC2007/JPEGImages/*.jpg   (or VOC2007/FOG/*.jpg for VOC-FOG)
<root>/VOC2007/Annotations/*.xml
<root>/VOC2007/ImageSets/Main/test.txt
```
This cell prints the tree of each attached dataset so we can confirm that layout (or spot where it differs) before running anything.

In [ ]:
import os

DATASET_BASES = {
    "fdd-dataset": "/kaggle/input/datasets/mdhabibourrahman/fdd-dataset/FDD",
    "voc-fog": "/kaggle/input/datasets/mdhabibourrahman/voc-fog/VOC_FOG",
    "rtts-dataset": "/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS",
    "rdfnet-baseline": "/kaggle/input/datasets/mdhabibourrahman/rdfnet-baseline",
}

def print_tree(root, max_depth=4, max_files_per_dir=10):
    root = root.rstrip('/')
    base_depth = root.count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        depth = dirpath.count(os.sep) - base_depth
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = '  ' * depth
        print(f"{indent}{os.path.basename(dirpath)}/")
        for f in sorted(filenames)[:max_files_per_dir]:
            print(f"{indent}  {f}")
        if len(filenames) > max_files_per_dir:
            print(f"{indent}  ... ({len(filenames) - max_files_per_dir} more files)")

for name, base in DATASET_BASES.items():
    print(f"\n===== {name}: {base} =====")
    if os.path.exists(base):
        print_tree(base)
    else:
        print("NOT FOUND -- check the exact path under /kaggle/input")

## Resolve each dataset's real layout

The tree printed above shows the three datasets are **not** laid out identically:
- **FDD**: `Annotations/` + `JPEGImages/` (`.jpg`) nested under an extra wrapper folder, no test-split file -- all images under it are the test set. Located by searching for the folder that actually contains both subdirs.
- **VOC-FOG**: has separate `train/` and `test/` folders. We evaluate on `test/`, which has `Annotations/` + `VOCtest-FOG/` (`.jpg`, foggy) -- no split file, so the id list comes from the images dir itself.
- **RTTS**: flat `Annotations/` + `ImageSets/Main/test.txt` (no `VOC2007` wrapper), but images in `JPEGImages/` are `.png`, not `.jpg`.

This cell builds the exact image-id list, images directory, and annotations directory for each, instead of assuming one common structure.

In [ ]:
import glob

def list_ids_from_dir(images_dir, ext):
    return sorted(os.path.splitext(f)[0] for f in os.listdir(images_dir) if f.lower().endswith(ext))

def read_test_ids(test_txt_path):
    return open(test_txt_path).read().strip().split()

def find_weight_file(base):
    if os.path.isfile(base) and base.endswith('.pth'):
        return base
    matches = glob.glob(os.path.join(base, '**', '*.pth'), recursive=True)
    if not matches:
        raise FileNotFoundError(f"No .pth weight file found under {base}.")
    return matches[0]

def find_data_root(base, required_subdirs):
    # dataset uploads have shifted nesting depth before (e.g. an extra FDD_DATASET/
    # wrapper folder) -- search instead of hardcoding a fixed depth.
    for dirpath, dirnames, _ in os.walk(base):
        if all(d in dirnames for d in required_subdirs):
            return dirpath
    raise FileNotFoundError(f"No directory under {base} contains all of {required_subdirs}")

# FDD: flat layout, no split file -- all images are the test set
FDD_BASE       = find_data_root(DATASET_BASES["fdd-dataset"], ("Annotations", "JPEGImages"))
FDD_IMAGES_DIR = os.path.join(FDD_BASE, "JPEGImages")
FDD_ANN_DIR    = os.path.join(FDD_BASE, "Annotations")
FDD_IDS        = list_ids_from_dir(FDD_IMAGES_DIR, ".jpg")

# VOC-FOG: dataset has separate train/ and test/ folders -- use test/ for evaluation.
# No split file here, so the id list comes straight from the foggy test images dir.
VOCFOG_ROOT       = find_data_root(DATASET_BASES["voc-fog"], ("Annotations", "VOCtest-FOG"))
VOCFOG_IMAGES_DIR = os.path.join(VOCFOG_ROOT, "VOCtest-FOG")
VOCFOG_ANN_DIR    = os.path.join(VOCFOG_ROOT, "Annotations")
VOCFOG_IDS        = list_ids_from_dir(VOCFOG_IMAGES_DIR, ".jpg")

# RTTS: flat layout (no VOC2007 wrapper), split defined by ImageSets/Main/test.txt, images are .png
RTTS_BASE       = DATASET_BASES["rtts-dataset"]
RTTS_IMAGES_DIR = os.path.join(RTTS_BASE, "JPEGImages")
RTTS_ANN_DIR    = os.path.join(RTTS_BASE, "Annotations")
RTTS_IDS        = read_test_ids(os.path.join(RTTS_BASE, "ImageSets", "Main", "test.txt"))

WEIGHT_PATH = find_weight_file(DATASET_BASES["rdfnet-baseline"])

print(f"FDD    : {len(FDD_IDS)} images    | images_dir={FDD_IMAGES_DIR}")
print(f"VOC-FOG: {len(VOCFOG_IDS)} images | images_dir={VOCFOG_IMAGES_DIR}")
print(f"RTTS   : {len(RTTS_IDS)} images   | images_dir={RTTS_IMAGES_DIR}")
print(f"WEIGHT_PATH: {WEIGHT_PATH}")

## Evaluation function

Same logic as the repo's `RDFNet/get_map.py`, parameterized so it can be called once per dataset instead of editing the script by hand each time.

In [ ]:
import xml.etree.ElementTree as ET
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from utils.utils import get_classes
from utils.utils_map import get_map
from yolo import YOLO

CLASSES_PATH = 'model_data/rtts_classes.txt'  # person, bicycle, car, motorbike, bus -- shared by all 3 datasets
ANCHORS_PATH = 'model_data/yolo_anchors.txt'  # YOLO's default anchors_path is a hardcoded Windows path from the original author's machine -- must override it

def evaluate(dataname, image_ids, images_dir, annotations_dir, model_path, image_ext='.jpg',
             classes_path=CLASSES_PATH, anchors_path=ANCHORS_PATH,
             min_overlap=0.5, confidence=0.001, nms_iou=0.5, score_threshold=0.5):
    map_out_path = f'/kaggle/working/map_out-{dataname}'
    for sub in ['ground-truth', 'detection-results', 'images-optional']:
        os.makedirs(os.path.join(map_out_path, sub), exist_ok=True)

    class_names, _ = get_classes(classes_path)
    print(f"[{dataname}] {len(image_ids)} images | loading model from {model_path}")
    yolo = YOLO(model_path=model_path, classes_path=classes_path, anchors_path=anchors_path,
                confidence=confidence, nms_iou=nms_iou, cuda=torch.cuda.is_available())

    skipped = []
    for image_id in tqdm(image_ids, desc=f'[{dataname}] predicting'):
        image_path = os.path.join(images_dir, image_id + image_ext)
        try:
            image = Image.open(image_path)
            image.load()  # force full decode now so truncated files raise here, not mid-inference
        except (UnidentifiedImageError, OSError) as e:
            # get_map() requires a detection-results file for every ground-truth file, so write
            # an empty one (== "zero detections") instead of crashing the whole evaluation run.
            skipped.append((image_id, str(e)))
            open(os.path.join(map_out_path, f"detection-results/{image_id}.txt"), "w").close()
            continue
        yolo.get_map_txt(image_id, image, class_names, map_out_path)

    if skipped:
        print(f"[{dataname}] WARNING: skipped {len(skipped)}/{len(image_ids)} unreadable image(s) "
              f"(treated as zero detections):")
        for image_id, err in skipped[:20]:
            print(f"    {image_id}{image_ext}: {err}")
        if len(skipped) > 20:
            print(f"    ... and {len(skipped) - 20} more")

    bad_xml = []
    for image_id in tqdm(image_ids, desc=f'[{dataname}] ground truth'):
        with open(os.path.join(map_out_path, f"ground-truth/{image_id}.txt"), "w") as new_f:
            try:
                root = ET.parse(os.path.join(annotations_dir, f"{image_id}.xml")).getroot()
            except ET.ParseError as e:
                # same flaky-upload risk as the images -- an empty ground-truth file just
                # means this image contributes 0 objects instead of crashing the whole run.
                bad_xml.append((image_id, str(e)))
                continue
            for obj in root.findall('object'):
                difficult_flag = obj.find('difficult') is not None and int(obj.find('difficult').text) == 1
                obj_name = obj.find('name').text
                if obj_name not in class_names:
                    continue
                bnd = obj.find('bndbox')
                left, top = bnd.find('xmin').text, bnd.find('ymin').text
                right, bottom = bnd.find('xmax').text, bnd.find('ymax').text
                suffix = " difficult" if difficult_flag else ""
                new_f.write(f"{obj_name} {left} {top} {right} {bottom}{suffix}\n")

    if bad_xml:
        print(f"[{dataname}] WARNING: {len(bad_xml)} unparseable annotation(s) (treated as 0 objects):")
        for image_id, err in bad_xml[:20]:
            print(f"    {image_id}.xml: {err}")

    print(f"[{dataname}] computing mAP")
    mAP = get_map(min_overlap, True, score_threhold=score_threshold, path=map_out_path)
    return mAP

## Run evaluation on all three datasets

In [ ]:
from PIL import Image, UnidentifiedImageError

def find_corrupt_images(dataname, image_ids, images_dir, image_ext):
    bad = []
    for image_id in tqdm(image_ids, desc=f'[{dataname}] integrity check'):
        image_path = os.path.join(images_dir, image_id + image_ext)
        try:
            with Image.open(image_path) as im:
                im.load()
        except (UnidentifiedImageError, OSError) as e:
            size = os.path.getsize(image_path) if os.path.exists(image_path) else -1
            bad.append((image_id, size, str(e)))
    return bad

for dataname, ids, images_dir, ext in [
    ('FDD', FDD_IDS, FDD_IMAGES_DIR, '.jpg'),
    ('VOC-FOG', VOCFOG_IDS, VOCFOG_IMAGES_DIR, '.jpg'),
    ('RTTS', RTTS_IDS, RTTS_IMAGES_DIR, '.png'),
]:
    bad = find_corrupt_images(dataname, ids, images_dir, ext)
    if bad:
        print(f"[{dataname}] {len(bad)}/{len(ids)} unreadable image(s):")
        for image_id, size, err in bad[:20]:
            print(f"    {image_id}{ext}  size={size}B  {err}")
        if len(bad) > 20:
            print(f"    ... and {len(bad) - 20} more")
    else:
        print(f"[{dataname}] all {len(ids)} images OK")

In [ ]:
results = {}
results['FDD'] = evaluate('fdd', FDD_IDS, FDD_IMAGES_DIR, FDD_ANN_DIR, WEIGHT_PATH, image_ext='.jpg')

In [ ]:
results['VOC-FOG'] = evaluate('VOC-FOG', VOCFOG_IDS, VOCFOG_IMAGES_DIR, VOCFOG_ANN_DIR, WEIGHT_PATH, image_ext='.jpg')

In [ ]:
results['RTTS'] = evaluate('rtts', RTTS_IDS, RTTS_IMAGES_DIR, RTTS_ANN_DIR, WEIGHT_PATH, image_ext='.png')

In [ ]:
print("\n=== RDFNet baseline -- mAP@0.5 summary ===")
for name, mAP in results.items():
    print(f"{name:8s}: {mAP:.2f}%")